In [13]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from deep_translator import GoogleTranslator
import scrapy
import time 
import pandas as pd

## No Fluff jobs Scrapper

In [15]:
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("--log-level=3")

driver = webdriver.Chrome(options=chrome_options)

SEARCH_KEYWORD = "python"

try:
    url = f"https://nofluffjobs.com/pl/jobs?criteria=keyword%3D'{SEARCH_KEYWORD}'"
    print(f"Searching NoFluffJobs for: {SEARCH_KEYWORD}")
    driver.get(url)

    wait = WebDriverWait(driver, 10)

    try:
        accept_btn = wait.until(EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler")))
        accept_btn.click()
        print("Cookies passed")
    except:
        print("No cookie pop-up")
    
    print("fetching job list....")
    cards = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a.posting-list-item")))

    job_links = [card.get_attribute("href") for card in cards[:5]]
    print(f"Found {len(job_links)} jobs. starting to extract requirements...")

    full_jobs_data = []

    translator = GoogleTranslator(source='auto', target='en')

    for link in job_links:
        try:
            driver.get(link)
            time.sleep(1)

            title = driver.find_element(By.TAG_NAME, "h1").text.strip()

            try:
                content = driver.find_element(By.TAG_NAME, "article").text
            except:
                content = driver.find_element(By.ID, "posting-description").text

            try:
                content_en = translator.translate(content[:4999])
            except:
                content_en = content
            full_jobs_data.append({
                "title": title,
                "link": link,
                "full_text": content_en,
                "source": "NoFluffJobs"
            })
            print(f" Scrapped -> {title}")
        except Exception as e:
            print(f" Failed -> {link} ({e})")

    df = pd.DataFrame(full_jobs_data)
    df.to_csv("scrapped_jobs_full.csv", index=False)
    print("Done ! data saved to a csv file")
    print(df.head())

finally:
    driver.quit()

Searching NoFluffJobs for: python
No cookie pop-up
fetching job list....
Found 5 jobs. starting to extract requirements...
No cookie pop-up
fetching job list....
Found 5 jobs. starting to extract requirements...
 Scrapped -> Senior Backend Engineer (Python)
 Scrapped -> Senior Backend Engineer (Python)
 Scrapped -> Senior AI Engineer (LLM | Langchain | Python)
 Scrapped -> Senior AI Engineer (LLM | Langchain | Python)
 Scrapped -> Staff Python Engineer
 Scrapped -> Staff Python Engineer
 Scrapped -> Senior Backend Developer (Python)
 Scrapped -> Senior Backend Developer (Python)
 Scrapped -> Python Backend Developer
Done ! data saved to a csv file
                                           title  \
0               Senior Backend Engineer (Python)   
1  Senior AI Engineer (LLM | Langchain | Python)   
2                          Staff Python Engineer   
3              Senior Backend Developer (Python)   
4                       Python Backend Developer   

                               